# XLS-R Nepali Baseline Audit (Colab)

Evaluates `gagan3012/wav2vec2-xlsr-nepali` on OpenSLR-43 (`gauravparajuli/slr43`)
-- the same corpus it was originally trained/self-evaluated on -- to sanity-check
its published **5.97% WER** claim.

**No GPU required** -- this is CTC greedy decoding (one forward pass per
utterance), not autoregressive generation. A free Colab CPU runtime is fine;
a GPU just makes it faster, especially for the full 2,064-utterance corpus.

Run the cells top to bottom. If you hit a dependency error after Step 2,
`Runtime -> Restart session` and re-run from Step 3 (not Step 2 again).

## Step 1: get the code

In [ ]:
import os

REPO_DIR = '/content/NSTT-Lite'
if not os.path.exists(REPO_DIR):
    !git clone -b plan4-continuation https://github.com/Rbimochan/NSTT-Lite.git {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull
%cd {REPO_DIR}

## Step 2: install dependencies
Self-contained (doesn't rely on `requirements.txt` being importable as a path) -- torch/torchaudio are intentionally NOT installed here, Colab already ships a GPU-matched pair.

In [ ]:
!pip install -q transformers==4.49.0 huggingface_hub==0.27.1 datasets==3.2.0 jiwer==3.0.5 soundfile==0.12.1
print('If this printed a red ResolutionImpossible error, stop and report it.')
print('Otherwise: Runtime -> Restart session now, then continue from Step 3 (skip Step 2 on the re-run).')

## Step 3: sanity-check the environment
Run this AFTER restarting the runtime post-install.

In [ ]:
import os, sys
REPO_DIR = '/content/NSTT-Lite'
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)

import torch, transformers, datasets
print('CUDA available:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (fine for this notebook)')
print('torch', torch.__version__, '| transformers', transformers.__version__, '| datasets', datasets.__version__)

## Step 4: load the model and dataset

In [ ]:
from src.xlsr_baseline import (
    load_xlsr_model_and_processor,
    load_openslr43_test_dataset,
    resample_if_needed,
    transcribe_ctc,
    build_openslr43_manifest_rows,
    write_openslr43_manifest,
    SELF_REPORTED_WER,
)
from src.wer_metrics import compute_wer_cer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Loading gagan3012/wav2vec2-xlsr-nepali (first run downloads ~1.2GB)...')
model, processor = load_xlsr_model_and_processor()
model.to(device)
print('Model loaded on', device)

In [ ]:
N_SAMPLES = 150  # set to None to run the full 2,064-utterance corpus (plan.md Step 1)

print('Loading gauravparajuli/slr43 (first run downloads ~935MB)...')
dataset = load_openslr43_test_dataset(max_samples=N_SAMPLES)
rows = build_openslr43_manifest_rows(dataset)
os.makedirs('data/manifests', exist_ok=True)
write_openslr43_manifest(rows, 'data/manifests/xlsr_openslr43_test_manifest.csv')
print(f'Loaded {len(dataset)} utterances.')

## Step 5: transcribe and score

In [ ]:
references, hypotheses = [], []
for i, row in enumerate(dataset):
    audio = row['audio']
    speech = resample_if_needed(audio['array'], audio['sampling_rate'])
    hyp = transcribe_ctc(model, processor, speech, device)
    references.append(rows[i]['text'])
    hypotheses.append(hyp)
    if (i + 1) % 25 == 0 or (i + 1) == len(dataset):
        print(f'{i + 1}/{len(dataset)} done')

wer, cer = compute_wer_cer(references, hypotheses)
print()
print(f'WER = {wer:.4f} ({wer*100:.2f}%)')
print(f'CER = {cer:.4f} ({cer*100:.2f}%)')
print(f'Self-reported WER = {SELF_REPORTED_WER:.4f} ({SELF_REPORTED_WER*100:.2f}%)')

## Step 6: save the result

In [ ]:
import json

results = {
    'model': 'gagan3012/wav2vec2-xlsr-nepali',
    'dataset': 'OpenSLR-43 (gauravparajuli/slr43)',
    'num_utterances': len(dataset),
    'wer': wer,
    'cer': cer,
    'self_reported_wer': SELF_REPORTED_WER,
    'note': (
        'Measured on a fixed-seed slice of the corpus (no independent held-out '
        'test split exists upstream). Single-speaker (female) corpus, no '
        'gender/demographic metadata -- no breakdown possible or attempted.'
        if N_SAMPLES is not None else
        'Measured on the full corpus (no independent held-out test split exists '
        'upstream). Single-speaker (female) corpus, no gender/demographic '
        'metadata -- no breakdown possible or attempted.'
    ),
}
os.makedirs('reports', exist_ok=True)
with open('reports/xlsr_baseline_results.json', 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=2, ensure_ascii=False)
print('Saved reports/xlsr_baseline_results.json:')
print(json.dumps(results, indent=2, ensure_ascii=False))

## Done

If you want to keep this result, download `reports/xlsr_baseline_results.json`
and `data/manifests/xlsr_openslr43_test_manifest.csv` from the Colab file
browser (left sidebar), or `git add`/`commit`/`push` from a terminal cell if
you've authenticated git in this session.